In [13]:
import os
import sys
import glob
import ctypes

# Preload pip-installed CUDA libraries before TensorFlow initializes in WSL.
cuda_libs = glob.glob(
    os.path.join(sys.prefix, "lib", "python*", "site-packages", "nvidia", "*", "lib", "*.so*")
)
for cuda_lib in cuda_libs:
    ctypes.CDLL(cuda_lib, mode=ctypes.RTLD_GLOBAL)

import tensorflow as tf
from tensorflow.keras import layers,Sequential

In [14]:
model=Sequential([
    layers.Input(shape=(150,150,3)),
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1,0.1),
  
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
# Classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),                   # prevent overfitting
    layers.Dense(1, activation='sigmoid')  # binary: cat or dog
])

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [16]:
full_ds = tf.keras.utils.image_dataset_from_directory(
    '/mnt/d/Datasets/PetImages',              # your folder name
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary',
    shuffle=True,
    seed=42
)

# 2. Split into train and validation (80/20)
total_batches = len(full_ds)
train_size = int(0.8 * total_batches)

train_ds = full_ds.take(train_size)

Found 24991 files belonging to 2 classes.


In [17]:
val_ds = full_ds.skip(train_size)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [18]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "cat_dog_best.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

model.save("cat_dog_final.keras")

Epoch 1/20


/mnt/d/DL-Algorithm/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1787857416.512631  632955 cuda_dnn.cc:461] Loaded cuDNN version 92400


 51/624 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.4841 - loss: 1.1106

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


109/624 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - accuracy: 0.5023 - loss: 0.8892

...

Corrupt JPEG data: 99 extraneous bytes before marker 0xd9


519/624 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.5010 - loss: 0.6932

623/624 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5015 - loss: 0.6932

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9
Corrupt JPEG data: 252 extraneous bytes before marker 0xd9
Corrupt JPEG data: 396 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9
Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 228 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1403 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1153 extraneous bytes before marker 0xd9
Corrupt JPEG data: 162 extraneous bytes before marker 0xd9
Corrupt JPEG data: 99 extraneous bytes before marker 0xd9
Corrupt JPEG data: 2226 extraneous bytes before marker 0xd9
Corrupt JPEG data: 65 extraneous bytes before marker 0xd9


624/624 ━━━━━━━━━━━━━━━━━━━━ 33s 52ms/step - accuracy: 0.5014 - loss: 0.6932 - val_accuracy: 0.4927 - val_loss: 0.6932 - learning_rate: 2.0000e-04
